# Private test — chạy suy luận bằng merged models
Kaggle: bật GPU, Internet OFF; attach đúng một project bundle mới, `en2vi_merged.zip`, `vi2en_merged.zip` và wheelhouse Linux Python 3.13. Không cần base model ZIP. Chạy trong session mới, Run All.
File nộp sau khi dịch đủ 100 tài liệu: `/kaggle/working/results_submission/results_private_context_v2/submission.zip`. Private không có reference nên không tính BLEU/chrF local.


In [ ]:
import os, sys, json, hashlib, zipfile, subprocess
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working/results_submission')
PROJECT_DIR = WORKING_ROOT / 'project'
WORKING_ROOT.mkdir(parents=True, exist_ok=True)
def safe_extract(path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()) or '\\' in member.filename:
                raise ValueError('Unsafe ZIP member')
        archive.extractall(destination)
# Kaggle may expose the uploaded archive unchanged, or automatically unpack
# it as a Dataset. Support both layouts and always copy into /kaggle/working.
bundles = list(INPUT_ROOT.rglob('NLP_results_submission_kaggle*.zip'))
manifests = []
for path in INPUT_ROOT.rglob('bundle_manifest.json'):
    try:
        metadata = json.loads(path.read_text(encoding='utf-8'))
        if metadata.get('purpose', '').startswith('Kaggle input for original full-document prediction'):
            manifests.append(path)
    except (OSError, UnicodeError, json.JSONDecodeError):
        pass
if PROJECT_DIR.exists() and not (PROJECT_DIR/'bundle_manifest.json').is_file():
    raise FileNotFoundError(f'Incomplete working project: {PROJECT_DIR}; remove it and rerun this cell')
if not PROJECT_DIR.exists():
    if len(bundles) == 1:
        safe_extract(bundles[0], WORKING_ROOT)
    elif len(bundles) == 0 and len(manifests) == 1:
        import shutil
        source_project = manifests[0].parent
        shutil.copytree(source_project, PROJECT_DIR)
    else:
        raise FileNotFoundError(
            f'Expected exactly one project ZIP or unpacked bundle_manifest.json; '
            f'ZIPs={bundles}, manifests={manifests}'
        )
manifest = json.loads((PROJECT_DIR/'bundle_manifest.json').read_text(encoding='utf-8'))
for relative, metadata in manifest['files'].items():
    path = (PROJECT_DIR / relative).resolve()
    if not path.is_relative_to(PROJECT_DIR.resolve()): raise ValueError('Unsafe bundle manifest')
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != metadata['sha256']:
        raise ValueError(f'Project file checksum mismatch: {relative}')
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_DATASETS_OFFLINE='1',
                  HF_HUB_DISABLE_TELEMETRY='1', TOKENIZERS_PARALLELISM='false',
                  CUBLAS_WORKSPACE_CONFIG=':4096:8')
sys.path.insert(0, str(PROJECT_DIR))
print('Project verified:', PROJECT_DIR)
print('Python:', sys.version)

Private test: 100 nguồn, 50 EN→VI và 50 VI→EN. Dùng hai merged model đã chốt, giữ config và context v2.


In [ ]:
wheel_dirs = sorted({p.parent for p in INPUT_ROOT.rglob('transformers-5.17.0-*.whl')})
if not wheel_dirs:
    wheel_zips = [p for p in INPUT_ROOT.rglob('*.zip') if 'wheelhouse' in p.name.lower()]
    if len(wheel_zips) != 1: raise FileNotFoundError(f'Expected one offline wheelhouse: {wheel_zips}')
    safe_extract(wheel_zips[0], WORKING_ROOT/'wheels')
    wheel_dirs = sorted({p.parent for p in (WORKING_ROOT/'wheels').rglob('transformers-5.17.0-*.whl')})
if len(wheel_dirs) != 1: raise ValueError(f'Ambiguous wheelhouse: {wheel_dirs}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheel_dirs[0]),
                '-r', str(PROJECT_DIR/'submission/requirements-inference.txt')], check=True)
# Kaggle's old torchao can fail while importing newer transformers.
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version
try:
    if Version(version('torchao')) <= Version('0.16.0'):
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
except PackageNotFoundError: pass
import torch
if not torch.cuda.is_available(): raise RuntimeError('Bật GPU ở Notebook Settings rồi restart session')
print('GPU:', torch.cuda.get_device_name(0))

Private test: 100 nguồn, 50 EN→VI và 50 VI→EN. Dùng hai merged model đã chốt, giữ config và context v2.


In [ ]:
from src.vinai_baseline.pipeline.promotion import file_hash
ADAPTER_DIRS = {}
for direction in ('en2vi', 'vi2en'):
    candidates = [p.parent for p in INPUT_ROOT.rglob('merge_manifest.json') if p.parent.name == direction]
    if not candidates:
        archives = list(INPUT_ROOT.rglob(f'{direction}_merged.zip'))
        if len(archives) != 1: raise ValueError(f'Expected one merged ZIP: {archives}')
        destination = WORKING_ROOT/'merged_models'
        safe_extract(archives[0], destination)
        candidates = [destination/direction]
    if len(candidates) != 1: raise ValueError(f'Ambiguous merged models: {candidates}')
    model_dir = candidates[0]
    metadata = json.loads((model_dir/'merge_manifest.json').read_text(encoding='utf-8'))
    config = json.loads((PROJECT_DIR/f'submission/configs/{direction}.json').read_text(encoding='utf-8'))
    assert (metadata['base_model'], metadata['base_model_revision'], metadata['direction']) == (config['model_name'], config['model_revision'], direction)
    for name, expected in metadata['files'].items():
        path = (model_dir/name).resolve()
        if not path.is_relative_to(model_dir.resolve()): raise ValueError('Unsafe manifest path')
        if path.stat().st_size != expected['bytes'] or file_hash(path) != expected['sha256']: raise ValueError(f'Model checksum mismatch: {path}')
    ADAPTER_DIRS[direction] = model_dir
print('Verified merged models:', ADAPTER_DIRS)


Private test: 100 nguồn, 50 EN→VI và 50 VI→EN. Dùng hai merged model đã chốt, giữ config và context v2.


In [ ]:
PHASE = 'private'
INPUT_CSV = PROJECT_DIR/'data/test/private_test_input.csv'
SOURCE_COLUMN = 'source_markdown'
BATCH_SIZE = 8  # Nếu OOM, dùng 4 và chọn RUN_DIR mới vì thay đổi batch ảnh hưởng provenance.
RUN_DIR = WORKING_ROOT/f'results_{PHASE}_context_v2'
from src.vinai_baseline.results_submission import load_test, validate_archive
documents = load_test(INPUT_CSV, PHASE, SOURCE_COLUMN, expected_documents=50 if PHASE=='public' else 100)
print('Expected prediction rows:', len(documents))
print('Per direction:', {d: sum(item.direction==d for item in documents) for d in ('en-vi', 'vi-en')})
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_results_submission', '-q'], cwd=PROJECT_DIR, check=True)
# Reuse the strong, already submitted EN-to-VI predictions when available.
# The archive is validated against all 100 required public keys before reuse.
REUSE_EN2VI_ZIP = None
if PHASE == 'public':
    previous = WORKING_ROOT/'results_public'/'submission.zip'
    candidates = [previous] if previous.is_file() else list(INPUT_ROOT.rglob('submission.zip'))
    valid = []
    for candidate in candidates:
        try:
            if validate_archive(candidate, documents)['rows'] == 100: valid.append(candidate)
        except Exception:
            pass
    if len(valid) == 1: REUSE_EN2VI_ZIP = valid[0]
    elif len(valid) > 1: raise ValueError(f'Nhiều baseline submission ZIP hợp lệ; chọn thủ công một file: {valid}')
print('Reuse locked EN-to-VI:', REUSE_EN2VI_ZIP or 'không có; sẽ sinh lại cả hai chiều')
def command(output, *extra):
    result = [sys.executable, '-u', str(PROJECT_DIR/'scripts/build_results_submission.py'),
              '--input', str(INPUT_CSV), '--phase', PHASE, '--output-dir', str(output),
              '--en2vi-model', str(ADAPTER_DIRS['en2vi']), '--vi2en-model', str(ADAPTER_DIRS['vi2en']),
              '--batch-size', str(BATCH_SIZE), '--resume', *extra]
    if SOURCE_COLUMN is not None: result += ['--source-column', SOURCE_COLUMN]
    if REUSE_EN2VI_ZIP is not None: result += ['--reuse-en2vi-zip', str(REUSE_EN2VI_ZIP)]
    return result

Private test: 100 nguồn, 50 EN→VI và 50 VI→EN. Dùng hai merged model đã chốt, giữ config và context v2.


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run(command(WORKING_ROOT/f'smoke_{PHASE}', '--smoke-documents', '1'), cwd=PROJECT_DIR, check=True)

In [ ]:
RUN_FULL = True
if RUN_FULL:
    if (RUN_DIR/'submission.zip').is_file():
        print('Đã có ZIP; chuyển sang cell kiểm tra và tải bên dưới.')
    else:
        subprocess.run(command(RUN_DIR), cwd=PROJECT_DIR, check=True)
else:
    print('Chưa dịch đầy đủ; không có bài nộp mới.')

Private test: 100 nguồn, 50 EN→VI và 50 VI→EN. Dùng hai merged model đã chốt, giữ config và context v2.


In [ ]:
from IPython.display import display, FileLink
from src.vinai_baseline.results_submission import validate_archive
ZIP_PATH = RUN_DIR/'submission.zip'
validation = validate_archive(ZIP_PATH, documents)
if validation['rows'] != 100: raise ValueError('Bài nộp phase hiện tại phải có đúng 100 dòng')
print(json.dumps(validation, indent=2))
report = json.loads((RUN_DIR/'evaluation_report.json').read_text(encoding='utf-8'))
print(json.dumps(report['metrics'], ensure_ascii=False, indent=2))
display(FileLink(str(ZIP_PATH)))
display(FileLink(str(RUN_DIR/'evaluation_report.json')))
display(FileLink(str(RUN_DIR/'results.csv')))
print('UPLOAD:', ZIP_PATH)